In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Install notebook dependencies in Colab runtime.
    %pip -q install rank-bm25 sentence-transformers rouge-score transformers torch

    from google.colab import drive
    drive.mount("/content/drive")
        

# Optional override if your repo root is non-standard.
# Example: /content/drive/MyDrive/Legal-question-answer-v1
PROJECT_ROOT = os.environ.get("PROJECT_ROOT", "")
if PROJECT_ROOT:
    print(f"Using PROJECT_ROOT from environment: {PROJECT_ROOT}")

In [ ]:
import sys
from pathlib import Path

# Resolve project root in both local and Colab environments.
candidate_roots = []

# 1) Explicit env override from setup cell
project_root_env = Path(PROJECT_ROOT).expanduser() if "PROJECT_ROOT" in globals() and PROJECT_ROOT else None
if project_root_env:
    candidate_roots.append(project_root_env)

# 2) Current working directory search upward
cwd = Path.cwd().resolve()
candidate_roots.extend([cwd, *cwd.parents])

# 3) Common Colab locations
candidate_roots.extend([
    Path("/content/drive/MyDrive/Colab Notebooks/NLP/Legal-question-answer-v1"),
])

project_root = None
for path in candidate_roots:
    if (path / "utils").exists() and (path / "data").exists():
        project_root = path
        break

if project_root is None:
    raise FileNotFoundError(
        "Could not find project root containing 'utils/' and 'data/'. "
        "Set PROJECT_ROOT env var, or update the Colab setup cell path."
    )

utils_path = str(project_root / "utils")
if utils_path not in sys.path:
    sys.path.append(utils_path)

print(f"Resolved project_root: {project_root}")

from get_jsonl_data import get_jsonl_data

In [ ]:
corpus_path = project_root / "data" / "corpus.jsonl"
chunk_store_path = project_root / "data" / "chunk_store.jsonl"

corpus = get_jsonl_data(str(corpus_path))
chunk_store = get_jsonl_data(str(chunk_store_path))

print(corpus[0])
print(chunk_store[0])

In [ ]:
import math
from collections import Counter
from rouge_score import rouge_scorer


rouge = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=False)

def normalize_whitespace(text):
    return " ".join((text or "").split())

def safe_bleu(reference_tokens, candidate_tokens, max_n=4, smooth=1.0):
    """
    Compatibility-safe BLEU implementation (no NLTK dependency).
    Returns a value in [0, 1].
    """
    if not reference_tokens or not candidate_tokens:
        return 0.0

    ref_len = len(reference_tokens)
    cand_len = len(candidate_tokens)
    if cand_len == 0:
        return 0.0

    clipped_precisions = []
    for n in range(1, max_n + 1):
        if cand_len < n:
            clipped_precisions.append(1e-12)
            continue

        ref_ngrams = Counter(tuple(reference_tokens[i:i + n]) for i in range(ref_len - n + 1))
        cand_ngrams = Counter(tuple(candidate_tokens[i:i + n]) for i in range(cand_len - n + 1))

        overlap = sum(min(count, ref_ngrams[ng]) for ng, count in cand_ngrams.items())
        total = sum(cand_ngrams.values())

        precision = (overlap + smooth) / (total + smooth) if smooth > 0 else (overlap / total if total else 0.0)
        clipped_precisions.append(max(precision, 1e-12))

    bp = 1.0 if cand_len > ref_len else math.exp(1.0 - (ref_len / cand_len))
    bleu = bp * math.exp(sum((1.0 / max_n) * math.log(p) for p in clipped_precisions))
    return float(max(0.0, min(1.0, bleu)))

def calculate_answer_score(answer, chunk, rouge_weight=0.6, bleu_weight=0.4):
    """
    Combined lexical score using ROUGE-L and BLEU.
    Returns a value in [0, 1].
    """
    answer = normalize_whitespace(answer)
    chunk = normalize_whitespace(chunk)

    if not answer or not chunk:
        return 0.0

    rouge_scores = rouge.score(answer, chunk)
    rouge_l_f1 = rouge_scores["rougeL"].fmeasure

    reference_tokens = answer.split()
    candidate_tokens = chunk.split()
    bleu = safe_bleu(reference_tokens, candidate_tokens)

    final_score = (rouge_weight * rouge_l_f1) + (bleu_weight * bleu)
    return float(final_score)

In [ ]:
import torch
from sentence_transformers import CrossEncoder

# Vietnamese-focused reranker (same family used in your hard-negative notebook).
reranker_model_name = "itdainb/PhoRanker"

device = "cuda" if torch.cuda.is_available() else "cpu"
cross_encoder = CrossEncoder(reranker_model_name, max_length=256, device=device)
print(f"Loaded reranker: {reranker_model_name} on {device}")

def truncate_pair_for_reranker(question, chunk, tokenizer, max_length):
    """
    Truncate only the chunk side so we preserve the full question as much as possible.
    This avoids tokenizer overflow warnings for pair truncation with longest_first.
    """
    question = (question or "").strip()
    chunk = (chunk or "").strip()
    if not question or not chunk:
        return question, chunk

    # Keep room for special tokens: [CLS] q [SEP] c [SEP].
    q_ids = tokenizer.encode(question, add_special_tokens=False)
    c_ids = tokenizer.encode(chunk, add_special_tokens=False)
    special_tokens = tokenizer.num_special_tokens_to_add(pair=True)

    # Reserve at least one token for chunk when possible.
    max_chunk_len = max(1, max_length - len(q_ids) - special_tokens)

    if len(c_ids) > max_chunk_len:
        c_ids = c_ids[:max_chunk_len]
        chunk = tokenizer.decode(c_ids, skip_special_tokens=True).strip()

    return question, chunk

def calculate_cross_encoder_scores(question, chunks, batch_size=16):
    """
    Batch cross-encoder relevance scores for (question, chunk) pairs.
    Returns normalized scores in [0, 1] with same length/order as chunks.
    """
    question = (question or "").strip()
    if not question or not chunks:
        return []

    scores = [0.0] * len(chunks)
    pairs = []
    valid_indices = []

    for idx, chunk in enumerate(chunks):
        chunk_text = (chunk or "").strip()
        if not chunk_text:
            continue

        q_text, c_text = truncate_pair_for_reranker(
            question=question,
            chunk=chunk_text,
            tokenizer=cross_encoder.tokenizer,
            max_length=cross_encoder.max_length,
        )
        if not c_text:
            continue

        pairs.append((q_text, c_text))
        valid_indices.append(idx)

    if not pairs:
        return scores

    raw_scores = cross_encoder.predict(
        pairs,
        batch_size=batch_size,
        show_progress_bar=False,
        convert_to_numpy=True
    )

    for idx, score in zip(valid_indices, raw_scores):
        scores[idx] = float(score)

    # Min-max normalize only over valid chunks for stable weighting.
    valid_scores = [scores[idx] for idx in valid_indices]
    min_score = min(valid_scores)
    max_score = max(valid_scores)

    if max_score > min_score:
        denom = max_score - min_score
        for idx in valid_indices:
            scores[idx] = (scores[idx] - min_score) / denom
    else:
        for idx in valid_indices:
            scores[idx] = 0.5

    return scores

In [ ]:
from collections import defaultdict
from rank_bm25 import BM25Okapi
import numpy as np

top_k = 50
top_k_pos = 5
answer_score_weight = 0.35
cross_encoder_score_weight = 0.65
progress_every_rows = 100

if answer_score_weight < 0 or cross_encoder_score_weight < 0:
    raise ValueError("Score weights must be non-negative.")
if not np.isclose(answer_score_weight + cross_encoder_score_weight, 1.0, atol=1e-6):
    raise ValueError("Score weights must sum to 1.0.")

# Pre-index chunks by article to avoid repeated linear scans.
chunks_by_article = defaultdict(list)
for chunk in chunk_store:
    article_id = chunk.get("article_id")
    chunk_text = normalize_whitespace(chunk.get("text"))
    if article_id is not None and chunk_text:
        chunks_by_article[article_id].append(chunk_text)

# Build BM25 once per article and cache it.
bm25_cache = {}
for article_id, chunk_texts in chunks_by_article.items():
    tokenized_chunks = [chunk.split() for chunk in chunk_texts]
    if tokenized_chunks:
        bm25_cache[article_id] = (chunk_texts, BM25Okapi(tokenized_chunks))

training_data = []

total_rows = len(corpus)
rows_with_cache = 0
qa_seen = 0
qa_valid = 0

print(f"Starting positive chunk generation for {total_rows} corpus rows...")

for row_idx, row in enumerate(corpus, start=1):
    doc_id = row.get("doc_id")
    article_id = row.get("article_id")
    qa_pairs = row.get("generated_qa_pairs", [])

    # Skip rows with no valid chunks for the article.
    if article_id not in bm25_cache:
        if row_idx % progress_every_rows == 0 or row_idx == total_rows:
            print(
                f"[Progress] rows={row_idx}/{total_rows} | rows_with_cache={rows_with_cache} | "
                f"qa_seen={qa_seen} | qa_valid={qa_valid} | samples={len(training_data)}"
            )
        continue

    rows_with_cache += 1
    chunk_texts, bm25 = bm25_cache[article_id]

    for qa_pair in qa_pairs:
        qa_seen += 1

        question = normalize_whitespace(qa_pair.get("question"))
        answer = normalize_whitespace(qa_pair.get("answer"))

        # Skip malformed QA entries.
        if not question or not answer:
            continue

        qa_valid += 1

        tokenized_question = question.split()
        bm25_scores = bm25.get_scores(tokenized_question)
        if len(bm25_scores) == 0:
            continue

        k = min(top_k, len(bm25_scores))
        if k == len(bm25_scores):
            top_k_idx = np.argsort(bm25_scores)[::-1]
        else:
            top_k_idx = np.argpartition(bm25_scores, -k)[-k:]
            top_k_idx = top_k_idx[np.argsort(bm25_scores[top_k_idx])[::-1]]

        top_k_chunks = [chunk_texts[idx] for idx in top_k_idx]
        answer_scores = [calculate_answer_score(answer, chunk) for chunk in top_k_chunks]
        cross_encoder_scores = calculate_cross_encoder_scores(question, top_k_chunks, batch_size=16)

        final_scores = []
        for chunk, answer_score, cross_encoder_score in zip(top_k_chunks, answer_scores, cross_encoder_scores):
            final_score = (answer_score * answer_score_weight) + (cross_encoder_score * cross_encoder_score_weight)
            final_scores.append((chunk, float(final_score)))

        final_scores.sort(key=lambda x: x[1], reverse=True)
        top_k_positive = final_scores[:top_k_pos]
        top_k_positive_chunks = [chunk for chunk, _ in top_k_positive]

        sample = {
            "doc_id": doc_id,
            "article_id": article_id,
            "question": question,
            "answer": answer,
            "positive_chunks": top_k_positive_chunks,
        }
        training_data.append(sample)

    if row_idx % progress_every_rows == 0 or row_idx == total_rows:
        print(
            f"[Progress] rows={row_idx}/{total_rows} | rows_with_cache={rows_with_cache} | "
            f"qa_seen={qa_seen} | qa_valid={qa_valid} | samples={len(training_data)}"
        )

print(
    f"Done. rows={total_rows}, rows_with_cache={rows_with_cache}, "
    f"qa_seen={qa_seen}, qa_valid={qa_valid}, samples={len(training_data)}"
)

In [ ]:
print("Number of training samples:", len(training_data))
if training_data:
    print(training_data[0])
else:
    print("No training samples were created. Check corpus/chunk quality and filtering conditions.")

In [ ]:
# Save the training data to a JSONL file
import json

output_path = project_root / "data" / "training_data.jsonl"
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    for sample in training_data:
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

print(f"Saved to: {output_path}")